# Supermarket Sales Analysis and Supervised Machine Learning
### Kaggle Supermarket Sales Dataset (1,000 Records)
---
## Project Executive Summary
1. Sales Trends across branches, cities, product lines.
2. Customer behavior by gender, membership, payment mode.
3. Supervised learning models: Decision Tree Classifier & Logistic Regression.
4. Model benchmark: Accuracy, Precision, Recall, F1-score, ROC-AUC.
5. Strategic business recommendations.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv("supermarket_sales.csv")
print("Shape:", df.shape)
df.head()


## 2. Preprocessing and Feature Engineering


In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.month_name()
df["Day"] = df["Date"].dt.day_name()
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Hour"] = pd.to_datetime(df["Time"], format="%H:%M").dt.hour
df["Rating_Category"] = (df["Rating"] >= 7.0).astype(int)
df["Rating_Label"] = df["Rating_Category"].map({1: "High (>= 7.0)", 0: "Low (< 7.0)"})
df.describe()


## 3. Exploratory Data Analysis (EDA)


In [ ]:
prod_summary = df.groupby("Product line").agg({"Total": "sum", "gross income": "sum", "Quantity": "sum", "Rating": "mean"}).sort_values(by="Total", ascending=False)
display(prod_summary.round(2))


In [ ]:
branch_summary = df.groupby(["Branch", "City"]).agg({"Total": ["sum", "mean", "count"], "Rating": "mean"}).round(2)
display(branch_summary)


In [ ]:
cust_summary = df.groupby(["Customer type", "Gender"]).agg({"Total": ["sum", "mean", "count"], "Rating": "mean"}).round(2)
display(cust_summary)


## 4. Supervised Machine Learning Pipeline


In [ ]:
feature_cols = ["Branch", "City", "Customer type", "Gender", "Product line", "Unit price", "Quantity", "Tax 5%", "Total", "Payment", "cogs", "gross income", "Hour", "DayOfWeek"]
df_encoded = df.copy()
cat_cols = ["Branch", "City", "Customer type", "Gender", "Product line", "Payment"]
for col in cat_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])
X = df_encoded[feature_cols]
y = df_encoded["Rating_Category"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train samples:", X_train.shape[0], "Test samples:", X_test.shape[0])


## 5. Model Training and Benchmarking


In [ ]:
dt_model = DecisionTreeClassifier(criterion="gini", max_depth=4, min_samples_split=10, min_samples_leaf=5, random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

lr_model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

benchmark_df = pd.DataFrame({
  "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
  "Decision Tree Classifier": [accuracy_score(y_test, y_pred_dt), precision_score(y_test, y_pred_dt), recall_score(y_test, y_pred_dt), f1_score(y_test, y_pred_dt), roc_auc_score(y_test, y_prob_dt)],
  "Logistic Regression": [accuracy_score(y_test, y_pred_lr), precision_score(y_test, y_pred_lr), recall_score(y_test, y_pred_lr), f1_score(y_test, y_pred_lr), roc_auc_score(y_test, y_prob_lr)]
})
display(benchmark_df.round(4))


## 6. Business Recommendations
1. **Focus on Leading Categories**: Food & Beverages (.1K) and Sports & Travel (.1K).
2. **Replicate Branch C Success**: Naypyitaw branch leads revenue (.57K) and AOV (.10).
3. **Incentivize Digital Payments**: Convert 34.4% cash shoppers into digital members via app cashback.
4. **Target High-Spending Female Members**: Female members lead in AOV (.73).
5. **Mitigate Peak Shopping Bottlenecks**: Increase cashier capacity during 1-3 PM and 6-8 PM peak hours.
